# Hard-row marks reader, phone-quality robust (run C)
**GPU (T4/P100) on, Internet on.** Upload the new `kaggle_marks.zip` (it now contains `data/cells_deg.npz`) as a new dataset version first.

Same specialist as before (warm start from `marks_sum.pt`, position channels, hard rows oversampled, synthetic hard rows), plus **phone-quality augmentation**: every epoch, each real train row is replaced half the time by a re-compressed / downsized copy of the same photo. Copies exist only for TRAIN sheets. Validation is scored on originals AND phone-quality copies of the val sheets (model selection uses both); the test sheets are scored once at the end, also on both.

Your current models are not touched: this writes `models/hard_c_*.pt`. `hard_b_s0.pt` stays as it is.


### 1. Setup (prints `True True True`)

In [ ]:
import glob, os, shutil, subprocess
os.makedirs('/kaggle/working/proj', exist_ok=True)
# Kaggle usually auto-extracts an uploaded zip, so look for the extracted folder first, then for the zip itself
hits = glob.glob('/kaggle/input/**/scripts/train_marks_sum.py', recursive=True)
if hits:
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
    print('copied extracted dataset from', src)
else:
    z = glob.glob('/kaggle/input/**/kaggle_marks.zip', recursive=True) + glob.glob('/kaggle/working/kaggle_marks.zip')
    assert z, 'Dataset not found: use "Add Input" on the right and attach your marks dataset, then re-run this cell.'
    subprocess.run(['unzip', '-qo', z[0], '-d', '/kaggle/working/proj'], check=True)
    print('unzipped', z[0])
os.chdir('/kaggle/working/proj')
print(os.path.exists('scripts/train_marks_hard.py'), os.path.exists('data/cells_deg.npz'), os.path.exists('models/marks_sum.pt'))


### 2. GPU check

In [ ]:
!nvidia-smi | head -15

### 3. Run C: specialist + phone-quality augmentation (seed 0)
Watch the `val-phone` columns and the `gap`. If `gap` keeps growing while `val` stops improving, it is memorising.

In [ ]:
!cd /kaggle/working/proj && python -u scripts/train_marks_hard.py --epochs 80 --seed 0 --coord --deg data/cells_deg.npz --out models/hard_c_s0.pt 2>&1 | grep --line-buffered -v -i warn | tee hard_c_s0.log

### 4. Result (paste this output to Claude)

In [ ]:
!grep -B1 -A22 'best checkpoint' hard_c_s0.log

### 5. Two more seeds (stability check)

In [ ]:
!cd /kaggle/working/proj && for s in 1 2; do python -u scripts/train_marks_hard.py --epochs 80 --seed $s --coord --deg data/cells_deg.npz --out models/hard_c_s$s.pt 2>&1 | grep --line-buffered -v -i warn | tee hard_c_s$s.log; done

In [ ]:
!grep -h -A14 'VALIDATION' hard_c_s1.log hard_c_s2.log

### 6. Download
Run, then use the Output panel (`/kaggle/working`) to download `marks_robust_models.zip`.

In [ ]:
import shutil, os, glob
os.chdir('/kaggle/working')
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in glob.glob('/kaggle/working/proj/models/hard_c_*'): shutil.copy(f, '/kaggle/working/out/')
for f in glob.glob('/kaggle/working/proj/hard_c_*.log'): shutil.copy(f, '/kaggle/working/out/')
shutil.make_archive('/kaggle/working/marks_robust_models', 'zip', '/kaggle/working/out')
print(round(os.path.getsize('marks_robust_models.zip')/1e6), 'MB')